## 1. What is a CSV file?

CSV stands for **Comma-Separated Values**. It is a plain-text format for storing table-like data in rows and columns.

Example:
```csv
OrderID,Product,Quantity,Price
1001,Notebook,2,50
1002,Pen,5,10
1003,Bag,1,800
```

The first row usually contains column names. Each later row is a record, and commas separate values.

### Real-life use cases
- **Retail:** sales, inventory, and order exports.
- **Hospitals:** appointment lists, billing exports, and operational reports.
- **Finance:** transaction exports and expense reports.
- **Education:** student marks, attendance, and survey responses.
- **Marketing:** campaign results, leads, and customer feedback.
- **Data science:** datasets for analysis and machine learning.

### Advantages
- Simple, portable, and widely supported.
- Can be opened by Excel, Google Sheets, Python, and many databases.
- Easy to transfer between systems.

### Limitations
- Does not preserve spreadsheet formatting, formulas, or multiple sheets.
- Does not formally store data types; software infers them when reading.
- Commas, quotes, and line breaks inside values need proper CSV quoting.
- Do not share confidential data unless you have permission and suitable safeguards.

# Pandas CSV Notes

**Topic:** Reading, exploring, cleaning, modifying, and exporting CSV files with Pandas  

## Learning objectives
- Understand CSV files and where they are used in real life.
- Load CSV files with `pd.read_csv()`.
- Inspect rows, columns, data types, and missing values.
- Select, filter, calculate, and summarize data.
- Handle common CSV problems.
- Save results using `DataFrame.to_csv()`.

## 2. Import Pandas

Pandas is a Python library for working with structured data. `pd` is its common alias.

If needed, install it in a notebook cell using `!pip install pandas`.

In [2]:
import pandas as pd
from io import StringIO

## 3. Create sample CSV data

In real work, you may receive a file such as `sales.csv`. To keep this notebook self-contained, we will create CSV text and read it as if it were a file.

Our sample represents a small shop's sales report.

In [21]:
csv_text = """OrderID,Date,Product,Category,Quantity,UnitPrice,City
1001,2026-09-01,Notebook,Stationery,2,50,Nashik
1002,2026-09-01,Pen,Stationery,5,10,Pune
1003,2026-09-02,Backpack,Bags,1,800,Mumbai
1004,2026-09-02,Notebook,Stationery,3,50,Nashik
1005,2026-09-03,Water Bottle,Accessories,2,250,Pune
1006,2026-09-03,Backpack,Bags,1,800,Mumbai
"""

sales = pd.read_csv(StringIO(csv_text))
sales

,OrderID,Product
0,0012,Pen
1,0013,Book


`StringIO(csv_text)` lets Pandas read a text string like a file. `pd.read_csv()` parses the CSV and returns a DataFrame.

For a real file saved on your computer, use:
```python
sales = pd.read_csv("sales.csv")
```
The path must point to the correct location. In Jupyter, relative paths are usually resolved from the notebook's working directory.

## 4. `pd.read_csv()` — reading CSV files

**Purpose:** Read a CSV file and convert it into a Pandas DataFrame.

Common syntax:
```python
pd.read_csv(
    filepath_or_buffer,
    sep=",",
    header="infer",
    names=None,
    usecols=None,
    dtype=None,
    parse_dates=None,
    na_values=None,
    encoding="utf-8"
)
```

You do not need to memorize every parameter at once.

### 4.1 `filepath_or_buffer`

The file path or file-like object to read.

```python
df = pd.read_csv("sales.csv")
df = pd.read_csv("data/sales.csv")
```

**Real-life use:** Load a sales export before calculating revenue.

### 4.2 `sep` — separator

CSV usually uses commas, but some exports use semicolons or tabs.

```python
df = pd.read_csv("sales_europe.csv", sep=";")
df = pd.read_csv("data.tsv", sep="\t")
```

**Real-life use:** If a regional accounting system uses semicolons and you leave the default comma separator, data may appear in one column.

### 4.3 `header` and `names`

- `header=0`: first row contains column names (the usual default).
- `header=None`: file has no header; Pandas assigns numeric column names.
- `names=[...]`: provide your own column names.

Example: a CSV with no header row.

In [7]:
headerless_text = """1001,Notebook,2
1002,Pen,5
1003,Bag,1
"""
headerless = pd.read_csv(
    StringIO(headerless_text),
    header=None,
    names=["OrderID", "Product", "Quantity"]
)
headerless

,OrderID,Product,Quantity
0,1001,Notebook,2
1,1002,Pen,5
2,1003,Bag,1


**Real-life use:** Older systems sometimes export records without column headings. `header=None` prevents the first record being treated as a header.

### 4.4 `usecols` — load only selected columns

```python
small_sales = pd.read_csv(
    StringIO(csv_text),
    usecols=["OrderID", "Product", "Quantity"]
)
small_sales
```

**Why use it?** It keeps analysis focused and can reduce memory use for large files.

**Real-life use:** An inventory export may contain many columns, but you only need product, quantity, and warehouse.

### 4.5 `dtype` — specify data types

Pandas infers data types, but you can explicitly set them.

```python
id_data = pd.read_csv(
    StringIO("OrderID,Product\n0012,Pen\n0013,Book\n"),
    dtype={"OrderID": "string"}
)
id_data
```

**Important:** IDs, postal codes, and product codes are often labels, not numbers for calculations. Reading them as strings preserves leading zeros.

**Real-life use:** Product code `00125` must not become `125` if leading zeros are meaningful.

### 4.6 `parse_dates` — parse dates

```python
sales_dates = pd.read_csv(
    StringIO(csv_text),
    parse_dates=["Date"]
)
sales_dates.dtypes
```

Parsed dates are easier to sort and analyze. Unusual date formats may need extra cleaning with `pd.to_datetime()`.

### 4.7 `na_values` — identify missing-value markers

CSV exports may use `NA`, `N/A`, `unknown`, or `-` to mean missing data.

```python
missing_text = '''Employee,Department,Bonus
Asha,Sales,5000
Ravi,IT,NA
Meera,HR,-
'''
employees = pd.read_csv(
    StringIO(missing_text),
    na_values=["NA", "-"]
)
employees
```

Pandas treats the specified markers as missing values.

**Caution:** Only mark a value as missing if it truly means missing in your dataset. A dash may be a valid code in some systems.

In [23]:
missing_text = '''Employee,Department,Bonus
Asha,Sales,5000
Ravi,IT,NA
Meera,HR,-
'''
employees = pd.read_csv(
    StringIO(missing_text),
    na_values=["NA", "-"]
)
employees

,Employee,Department,Bonus
0,Asha,Sales,5000.0
1,Ravi,IT,NaN
2,Meera,HR,NaN


## 5. Inspect a newly loaded CSV

Inspect data before analyzing it.

| Command | What it tells you |
|---|---|
| `df.head()` | First five rows |
| `df.head(10)` | First ten rows |
| `df.tail()` | Last five rows |
| `df.shape` | Number of rows and columns |
| `df.columns` | Column labels |
| `df.dtypes` | Data type of each column |
| `df.info()` | Column types and non-null counts |
| `df.describe()` | Summary statistics for numeric columns |
| `df.describe(include="all")` | Summary for numeric and non-numeric columns |

In [24]:
sales.head(3)

,OrderID,Date,Product,Category,Quantity,UnitPrice,City
0,1001,2026-09-01,Notebook,Stationery,2,50,Nashik
1,1002,2026-09-01,Pen,Stationery,5,10,Pune
2,1003,2026-09-02,Backpack,Bags,1,800,Mumbai


In [25]:
sales.shape

(6, 7)

In [27]:
sales.columns

Index(['OrderID', 'Date', 'Product', 'Category', 'Quantity', 'UnitPrice',
       'City'],
      dtype='str')

In [28]:
sales.dtypes

OrderID      int64
Date           str
Product        str
Category       str
Quantity     int64
UnitPrice    int64
City           str
dtype: object

In [12]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   OrderID    6 non-null      int64
 1   Date       6 non-null      str  
 2   Product    6 non-null      str  
 3   Category   6 non-null      str  
 4   Quantity   6 non-null      int64
 5   UnitPrice  6 non-null      int64
 6   City       6 non-null      str  
dtypes: int64(3), str(4)
memory usage: 312.0 bytes


In [30]:
sales.describe(include="all")

,OrderID,Date,Product,Category,Quantity,UnitPrice,City
count,6.000000,6,6,6,6.000000,6.000000,6
unique,NaN,3,4,3,NaN,NaN,3
top,NaN,2026-09-01,Notebook,Stationery,NaN,NaN,Nashik
freq,NaN,2,2,3,NaN,NaN,2
mean,1003.500000,NaN,NaN,NaN,2.333333,326.666667,NaN
std,1.870829,NaN,NaN,NaN,1.505545,376.120548,NaN
min,1001.000000,NaN,NaN,NaN,1.000000,10.000000,NaN
25%,1002.250000,NaN,NaN,NaN,1.250000,50.000000,NaN
50%,1003.500000,NaN,NaN,NaN,2.000000,150.000000,NaN
75%,1004.750000,NaN,NaN,NaN,2.750000,662.500000,NaN


**Real-life example:** After importing an appointment export, checking `shape`, `dtypes`, and `info()` can reveal an empty file, unexpected columns, or dates read as ordinary text.

## 6. Select columns and rows

### One column
```python
sales["Product"]
```
Returns a Series.

### Multiple columns
```python
sales[["Product", "Quantity", "UnitPrice"]]
```
Returns a DataFrame.

### Rows by position with `iloc`
```python
sales.iloc[0]      # first row
sales.iloc[0:3]    # first three rows
```

### Rows by condition with `loc`
```python
sales.loc[sales["City"] == "Nashik"]
```

**Real-life use:** Find orders from a particular city, students with low marks, or products from a selected category.

In [49]:
sales["Product"]

0        Notebook
1             Pen
2        Backpack
3        Notebook
4    Water Bottle
5        Backpack
Name: Product, dtype: str

In [15]:
sales[["Product", "Quantity", "UnitPrice"]]

,Product,Quantity,UnitPrice
0,Notebook,2,50
1,Pen,5,10
2,Backpack,1,800
3,Notebook,3,50
4,Water Bottle,2,250
5,Backpack,1,800


In [16]:
sales.loc[sales["City"] == "Nashik"]

,OrderID,Date,Product,Category,Quantity,UnitPrice,City
0,1001,2026-09-01,Notebook,Stationery,2,50,Nashik
3,1004,2026-09-02,Notebook,Stationery,3,50,Nashik


## 7. Filter records with conditions

Conditions evaluate to `True` or `False` for each row. Use `&` for AND and `|` for OR; put each condition in parentheses.

```python
sales.loc[sales["Quantity"] >= 2]
sales.loc[(sales["City"] == "Pune") & (sales["Quantity"] >= 2)]
sales.loc[sales["City"].isin(["Pune", "Mumbai"])]
```

**Real-life examples**
- Retail: find products with low stock.
- Education: find students below a pass threshold.
- Operations: filter orders by location.
- Customer support: filter tickets by status or priority.

In [51]:
sales.loc[sales["Quantity"] >= 2]

,OrderID,Date,Product,Category,Quantity,UnitPrice,City
0,1001,2026-09-01,Notebook,Stationery,2,50,Nashik
1,1002,2026-09-01,Pen,Stationery,5,10,Pune
3,1004,2026-09-02,Notebook,Stationery,3,50,Nashik
4,1005,2026-09-03,Water Bottle,Accessories,2,250,Pune


In [58]:
sales.loc[(sales["City"] == "Pune") & (sales["Quantity"] >= 2)]

,OrderID,Date,Product,Category,Quantity,UnitPrice,City
1,1002,2026-09-01,Pen,Stationery,5,10,Pune
4,1005,2026-09-03,Water Bottle,Accessories,2,250,Pune


## 8. Add calculated columns

You can calculate a new column from existing columns. For this sales example, line revenue is `Quantity × UnitPrice`.

```python
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales
```

This changes the DataFrame in memory, not the original CSV file on disk.

**Real-life use cases**
- Retail: revenue = quantity × unit price.
- Education: total marks or percentage.
- HR: tenure from employment dates.
- Logistics: estimated cost from distance and rate.

Confirm the business definition first: discounts, refunds, taxes, and shipping can affect real revenue.

In [59]:
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales

,OrderID,Date,Product,Category,Quantity,UnitPrice,City,Revenue
0,1001,2026-09-01,Notebook,Stationery,2,50,Nashik,100
1,1002,2026-09-01,Pen,Stationery,5,10,Pune,50
2,1003,2026-09-02,Backpack,Bags,1,800,Mumbai,800
3,1004,2026-09-02,Notebook,Stationery,3,50,Nashik,150
4,1005,2026-09-03,Water Bottle,Accessories,2,250,Pune,500
5,1006,2026-09-03,Backpack,Bags,1,800,Mumbai,800


## 9. Summarize data with `groupby()`

`groupby()` groups records, applies a calculation, and combines the results.

```python
sales.groupby("City")["Revenue"].sum()
sales.groupby("Product")["Quantity"].sum()
sales.groupby("Category")["Revenue"].mean()
```

**Real-life use:** Compare sales by city, identify popular products, or determine which category contributes the most revenue.

In [60]:
sales.groupby("City")["Revenue"].sum()

City
Mumbai    1600
Nashik     250
Pune       550
Name: Revenue, dtype: int64

In [68]:
sales.groupby("Product")["Quantity"].sum().sort_values(ascending=True)

Product
Backpack        2
Water Bottle    2
Notebook        5
Pen             5
Name: Quantity, dtype: int64

## 10. Sort records with `sort_values()`

```python
sales.sort_values("Revenue", ascending=False)
```

- `ascending=True`: smallest to largest (default).
- `ascending=False`: largest to smallest.

**Real-life use:** Show high-value orders first, rank students by marks, or find products with the most units sold.

In [69]:
sales.sort_values("Revenue", ascending=False)

,OrderID,Date,Product,Category,Quantity,UnitPrice,City,Revenue
2,1003,2026-09-02,Backpack,Bags,1,800,Mumbai,800
5,1006,2026-09-03,Backpack,Bags,1,800,Mumbai,800
4,1005,2026-09-03,Water Bottle,Accessories,2,250,Pune,500
3,1004,2026-09-02,Notebook,Stationery,3,50,Nashik,150
0,1001,2026-09-01,Notebook,Stationery,2,50,Nashik,100
1,1002,2026-09-01,Pen,Stationery,5,10,Pune,50


## 11. Handle missing values

Useful methods:
- `df.isna()` — `True` where a value is missing.
- `df.isna().sum()` — missing-value count per column.
- `df.dropna()` — remove rows containing missing values by default.
- `df.fillna(value)` — fill missing values.

Create a small dataset with a missing salary and a duplicate record:

In [80]:
quality_text = """Employee,Department,Salary
Asha,Sales,40000
Ravi,IT,
Meera,HR,45000
Asha,Sales,40000
"""
quality = pd.read_csv(StringIO(quality_text))
quality

,Employee,Department,Salary
0,Asha,Sales,40000.0
1,Ravi,IT,NaN
2,Meera,HR,45000.0
3,Asha,Sales,40000.0


In [24]:
quality.isna().sum()

Employee      0
Department    0
Salary        1
dtype: int64

You could fill missing numeric values with the median if appropriate:

```python
quality["Salary"] = quality["Salary"].fillna(quality["Salary"].median())
```

Or remove rows missing a salary:
```python
quality_without_missing_salary = quality.dropna(subset=["Salary"])
```

**Caution:** Do not automatically replace every missing value with zero. Zero may mean something different from “not recorded.” Choose a method based on the data and goal.

In [25]:
quality["Salary"] = quality["Salary"].fillna(quality["Salary"].median())
quality

,Employee,Department,Salary
0,Asha,Sales,40000.0
1,Ravi,IT,40000.0
2,Meera,HR,45000.0
3,Asha,Sales,40000.0


## 12. Remove duplicate rows with `drop_duplicates()`

Repeated records can distort totals and counts.

```python
quality.duplicated().sum()  # count exact duplicate rows
quality.drop_duplicates()   # return rows without exact duplicates
```

To update the variable:
```python
quality = quality.drop_duplicates()
```

**Real-life use:** Remove duplicate product records or repeated survey submissions after confirming what counts as a duplicate.

**Important:** Identical-looking rows are not always accidental duplicates. For business data, check the transaction ID or other appropriate key before deleting records.

In [89]:
quality.duplicated().sum()

np.int64(1)

In [88]:
quality.drop_duplicates()

np.int64(1)

## 13. Convert column data types

Sometimes a CSV column contains numbers or dates as text.

### Convert to numeric
```python
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
```
Invalid values become missing values; inspect them afterward.

### Convert to dates
```python
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
```

**Real-life use:** Convert exported dates before preparing monthly reports, or convert amounts before calculating totals.

## 14. Clean text values

CSV exports may contain extra spaces or inconsistent capitalization.

```python
sales["City"] = sales["City"].str.strip()
sales["Product"] = sales["Product"].str.strip()
```

Other useful operations:
```python
df["Name"].str.lower()
df["Name"].str.upper()
df["Name"].str.contains("pen", case=False, na=False)
```

**Real-life use:** `"Pune"`, `" Pune "` and `"PUNE"` may refer to the same city. Normalize text before grouping, and keep the original data if you need an audit trail.

## 15. Export a DataFrame with `to_csv()`

After cleaning or analyzing data, save the result.

**Syntax**
```python
df.to_csv("output.csv", index=False)
```

- `index=False` prevents the DataFrame index from being written as an extra column.
- If you omit it, the index is written by default.
- The path can include a folder name.

Example:
```python
sales.to_csv("cleaned_sales.csv", index=False)
```

**Real-life use:** Export a cleaned report for Excel users, share a summary with another system, or save a prepared dataset for later analysis.

> The export creates or overwrites a file at the specified path. Check the path and avoid overwriting important source files.

In [90]:
sales.to_csv("cleaned_sales.csv", index=False)

### Read the exported file back in

```python
saved_sales = pd.read_csv("cleaned_sales.csv")
```

Run the export first to create the file.

## 16. Common CSV problems and solutions

| Problem | Possible solution |
|---|---|
| File not found | Check the filename, folder, and working directory. |
| All data appears in one column | Check the delimiter with `sep`. |
| Leading zeros disappear | Use `dtype={"ID": "string"}`. |
| Dates are treated as text | Use `parse_dates` or `pd.to_datetime()`. |
| Strange characters appear | Check encoding; `encoding="utf-8-sig"` helps with some spreadsheet exports. |
| Missing values are not detected | Use `na_values` for known missing markers. |
| An unwanted index column appears after export | Use `to_csv(..., index=False)`. |
| Amounts contain currency symbols or commas | Clean strings carefully before numeric conversion. |

**Encoding note:** Prefer the encoding expected by the source file. UTF-8 is a good starting point; do not change encoding randomly.

## 17. End-to-end mini project: shop sales report

**Scenario:** A shop owner wants total revenue by city and the products with the most units sold.

Workflow:
1. Read the CSV.
2. Inspect the data.
3. Check missing values and duplicates.
4. Create a revenue column.
5. Summarize results.
6. Optionally export reports.

In [106]:
# Start from the original sample CSV
report = pd.read_csv(StringIO(csv_text))

# Inspect the data
print("Shape:", report.shape)
print("Missing values per column:")
print(report.isna().sum())
print("Duplicate rows:", report.duplicated().sum())

# Convert date and calculate revenue
report["Date"] = pd.to_datetime(report["Date"], errors="coerce")
report["Revenue"] = report["Quantity"] * report["UnitPrice"]

# Summarize
revenue_by_city = report.groupby("City", as_index=False)["Revenue"].sum()
units_by_product = (
    report.groupby("Product", as_index=False)["Quantity"]
    .sum()
    .sort_values("Quantity", ascending=False)
)

print("\nRevenue by city:")
display(revenue_by_city)

print("\nUnits sold by product:")
display(units_by_product)

# Optional: uncomment to create CSV files
# report.to_csv("cleaned_sales.csv", index=False)
# revenue_by_city.to_csv("revenue_by_city.csv", index=False)
# units_by_product.to_csv("units_by_product.csv", index=False)

Shape: (6, 7)
Missing values per column:
OrderID      0
Date         0
Product      0
Category     0
Quantity     0
UnitPrice    0
City         0
dtype: int64
Duplicate rows: 0

Revenue by city:


,City,Revenue
0,Mumbai,1600
1,Nashik,250
2,Pune,550



Units sold by product:


,Product,Quantity
1,Notebook,5
2,Pen,5
0,Backpack,2
3,Water Bottle,2


## 18. Practice exercises

Try these in new code cells:

1. Read the sample CSV into a DataFrame called `df`.
2. Display only `Product` and `Quantity`.
3. Find orders where `UnitPrice` is greater than `100`.
4. Add a `Revenue` column.
5. Calculate total revenue by `Category`.
6. Sort orders by `Quantity`, highest first.
7. Check for missing values.
8. Export the DataFrame to `practice_output.csv` without the index.

**Hint for exercise 5**
```python
df.groupby("Category")["Revenue"].sum()
```

## Quick revision

- **Read:** `pd.read_csv("file.csv")`
- **Inspect:** `head()`, `shape`, `columns`, `dtypes`, `info()`, `describe()`
- **Select:** `df["Column"]`, `df[["A", "B"]]`, `loc`, `iloc`
- **Filter:** `df.loc[condition]`
- **Clean:** `isna()`, `fillna()`, `dropna()`, `drop_duplicates()`
- **Transform:** `to_numeric()`, `to_datetime()`, string methods
- **Analyze:** `groupby()`, `sum()`, `mean()`, `sort_values()`
- **Save:** `df.to_csv("output.csv", index=False)`

## Official documentation
- [Read CSV files — `read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [Write CSV files — `to_csv`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
- [Pandas getting-started tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html)